In [ ]:
"""Entity-token transformer policy trained with Differentiable Predictive Control
through a learned world model (Lorenzo Giannuzzo, Politecnico di Torino).
"""

import glob
import os
import sys

COMPETITION_DATA_FOLDER = '/kaggle/input/competitions/neurips-2026-smart-buildings/'
sys.path.append(os.path.join(COMPETITION_DATA_FOLDER, 'kaggle_evaluation'))
sys.path.append(COMPETITION_DATA_FOLDER)

#Lorenzo Giannuzzo: the dataset mount point depends on the Kaggle layout, so
#the bundle is located by its marker file instead of a fixed path.
marker = glob.glob('/kaggle/input/**/SBWM_POLICY_BUNDLE', recursive=True)
assert marker, 'attach the sbwm-policy dataset to this notebook'
BUNDLE = os.path.dirname(marker[0])
sys.path.insert(0, BUNDLE)

import numpy as np
from kaggle_evaluation import smart_buildings_inference_server
from sbwm.policy import DPCPolicy

WEIGHTS = os.path.join(BUNDLE, 'policy.pt')


class TransferPolicy(smart_buildings_inference_server.Policy):
    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        self.inner = DPCPolicy(building_meta_data, action_spec, weights=WEIGHTS)
        c, d = action_spec['continuous_action'], action_spec['discrete_action']
        self.c_low, self.c_high = c.low, c.high
        self.d_low, self.d_high = d.low, d.high
        print(f'Controlling a building with {self.inner.v} zones', flush=True)

    def get_action(self, obs: np.ndarray) -> dict[str, np.ndarray]:
        a = self.inner.get_action(obs)
        #Lorenzo Giannuzzo: actions are clipped into the spec bounds, since the
        #gateway rejects out-of-range actions.
        return {
            'continuous_action': np.clip(np.asarray(a['continuous_action'], np.float32),
                                         self.c_low, self.c_high).astype(np.float32),
            'discrete_action': np.clip(np.asarray(a['discrete_action']),
                                       self.d_low, self.d_high).astype(np.int32),
        }


smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run(
    competition_data_folder=COMPETITION_DATA_FOLDER,
)
